# FoxTrack 03: Supervised models, leave one capture out

This notebook trains supervised models (logistic regression and gradient-boosted trees, Spark MLlib) on eight captures and tests each model on the ninth capture.

**Before you run it:** add the output of notebook 01 in **Input > Add Input > Notebook Output**, and set **Internet** to on.

**Why this design.** Each capture has one attack start. A model that trains and tests on the same capture cannot measure the delay at that start, because the start would be in the training data. Here the test capture is new to the model, so the result shows how well the model detects an attack type that it did not see.

**Method.**

1. Training data: a random sample of each capture, at most `N_PER_CLASS` benign and `N_PER_CLASS` malicious packets.
2. For each capture: train on the samples of the other eight captures, then score the test capture.
3. Threshold: from benign packets of the test capture (the same range as in notebook 02). The results are thus comparable with the baseline.

In [1]:
# Settings.
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
OUT_DIR = "/kaggle/working/supervised"
N_PER_CLASS = 100_000                  # sample size for each class and capture
MODELS = {"logreg": {}, "gbt": {"maxIter": 50, "maxDepth": 5}}
TARGET_FPRS = (1e-2, 1e-3, 1e-4)
ALERT_RULES = ((1, 1), (3, 5), (10, 20))

In [2]:
import shutil, subprocess, sys
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
!pip install -q pyspark mlflow

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
# A new clone on each run, so the code is always the latest version.
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
done = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
     f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
if done.returncode:
    raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import etl, split, supervised
print("Code version:", subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

Code version: 91c856c Add supervised leave-one-capture-out experiment


In [3]:
# Find the Parquet output of notebook 01.
from pathlib import Path
found = sorted(Path("/kaggle/input").rglob("summary.json")) + sorted(Path("/kaggle/working").glob("*/summary.json"))
if not found:
    raise FileNotFoundError("summary.json not found. Add the output of notebook 01 in Input > Add Input > Notebook Output.")
PARQUET_ROOT = found[0].parent
print("Parquet data:", PARQUET_ROOT)

Parquet data: /kaggle/input/notebooks/<kaggle-user>/notebookabc04dfe5a/kitsune_parquet


## 1. Make the training sample

In [4]:
%%time
import os
import mlflow
import pandas as pd

spark = etl.build_spark(driver_memory="20g")
summary = split.load_summary(PARQUET_ROOT)
os.makedirs(OUT_DIR, exist_ok=True)

sample_path = f"{OUT_DIR}/train_sample"
supervised.build_training_sample(spark, PARQUET_ROOT, summary, N_PER_CLASS).write.mode("overwrite").parquet(sample_path)
train = supervised.assemble(spark.read.parquet(sample_path)).cache()
train.groupBy("attack").pivot("label").count().orderBy("attack").show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 23:45:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/21 23:45:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------+------+------+
|           attack|     0|     1|
+-----------------+------+------+
|   active_wiretap|100000|100000|
|         arp_mitm|100000|100000|
|          fuzzing|100000|100000|
|     mirai_botnet|100000|100000|
|          os_scan|100000| 65700|
|       ssdp_flood|100000|100000|
|ssl_renegotiation|100000| 92652|
|          syn_dos|100000|  7038|
|  video_injection|100000|100000|
+-----------------+------+------+

CPU times: user 3.89 s, sys: 532 ms, total: 4.42 s
Wall time: 2min 28s


## 2. Train and test each fold

In [5]:
%%time
import time
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment("supervised_leave_one_out")
os.makedirs(f"{OUT_DIR}/scores", exist_ok=True)

all_results = []
for row in summary:
    s = split.onset_split(row)
    capture = spark.read.parquet(f"{PARQUET_ROOT}/features/attack={s.attack}")
    for name, params in MODELS.items():
        start = time.time()
        results, test = supervised.run_fold(train, capture, s, name, TARGET_FPRS, ALERT_RULES, **params)
        all_results.append(results)
        test.to_parquet(f"{OUT_DIR}/scores/{name}_{s.attack}.parquet", index=False)

        with mlflow.start_run(run_name=f"{name}_{s.attack}"):
            mlflow.log_params({"test_capture": s.attack, "model": name, "n_per_class": N_PER_CLASS, **params})
            mlflow.log_metric("roc_auc", results["roc_auc"].iloc[0])
            mlflow.log_metric("pr_auc", results["pr_auc"].iloc[0])
            for r in results.itertuples():
                tag = f"fpr{r.target_fpr:g}_k{r.k}n{r.n}"
                mlflow.log_metric(f"false_alarms_per_100k_{tag}", r.false_alarms_per_100k)
                mlflow.log_metric(f"packet_recall_{tag}", r.packet_recall)
                if r.detected:
                    mlflow.log_metric(f"delay_packets_{tag}", r.delay_packets)
        print(f"{s.attack} / {name}: done in {time.time() - start:.0f} s")

results = pd.concat(all_results, ignore_index=True)
results.to_csv(f"{OUT_DIR}/results.csv", index=False)

2026/11/21 23:48:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/21 23:48:03 INFO mlflow.store.db.utils: Updating database tables
2026/11/21 23:48:08 INFO mlflow.tracking.fluent: Experiment with name 'supervised_leave_one_out' does not exist. Creating a new experiment.


arp_mitm / logreg: done in 88 s
arp_mitm / gbt: done in 264 s
active_wiretap / logreg: done in 71 s
active_wiretap / gbt: done in 262 s
fuzzing / logreg: done in 72 s
fuzzing / gbt: done in 251 s
mirai_botnet / logreg: done in 62 s
mirai_botnet / gbt: done in 222 s
os_scan / logreg: done in 70 s
os_scan / gbt: done in 246 s
ssdp_flood / logreg: done in 85 s
ssdp_flood / gbt: done in 244 s
ssl_renegotiation / logreg: done in 70 s
ssl_renegotiation / gbt: done in 237 s
syn_dos / logreg: done in 66 s
syn_dos / gbt: done in 249 s
video_injection / logreg: done in 67 s
video_injection / gbt: done in 236 s
CPU times: user 2min 24s, sys: 22.5 s, total: 2min 46s
Wall time: 47min 50s


<timed exec>:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


## 3. Results

**Ranking quality** on the test capture. These numbers do not use a threshold. A ROC-AUC near 0.5 shows that the model cannot separate the new attack from benign traffic.

In [6]:
results.groupby(["attack", "model"])[["roc_auc", "pr_auc"]].first().unstack("model").round(3)

roc_auc        pr_auc       
model                 gbt logreg    gbt logreg
attack                                        
active_wiretap      0.771  0.422  0.929  0.705
arp_mitm            0.980  0.689  0.995  0.890
fuzzing             0.197  0.400  0.401  0.306
mirai_botnet        0.067  0.065  0.927  0.923
os_scan             0.801  0.240  0.432  0.074
ssdp_flood          0.963  0.044  0.976  0.662
ssl_renegotiation   0.892  0.830  0.822  0.521
syn_dos             0.857  0.924  0.244  0.217
video_injection     0.441  0.435  0.150  0.154

**Detection delay in packets** at a target false positive rate of 0.001 (empty = not detected). Columns are the model and the alert rule `k/n`.

In [7]:
results["rule"] = results["k"].astype(str) + "/" + results["n"].astype(str)
at = results[results["target_fpr"] == 1e-3]
at.pivot(index="attack", columns=["model", "rule"], values="delay_packets")

model               logreg                         gbt                  
rule                   1/1      3/5     10/20      1/1      3/5    10/20
attack                                                                  
active_wiretap       121.0    123.0     131.0      0.0      0.0      0.0
arp_mitm            8495.0   8519.0    8555.0      0.0      0.0      0.0
fuzzing             3857.0  12854.0   12861.0   1028.0      NaN      NaN
mirai_botnet          37.0    149.0    1045.0     10.0     22.0   9591.0
os_scan               29.0   3838.0  180571.0  12042.0  12045.0  12060.0
ssdp_flood             0.0      0.0    7570.0      0.0      0.0   4585.0
ssl_renegotiation    439.0    446.0    4242.0   1505.0   3207.0  59278.0
syn_dos              523.0    549.0   27238.0   9660.0  27231.0  27238.0
video_injection    78139.0  78141.0   78148.0      NaN      NaN      NaN

**False alarms before the attack start**, for each 100,000 benign packets, at the same target rate.

In [8]:
at.pivot(index="attack", columns=["model", "rule"], values="false_alarms_per_100k").round(1)

model              logreg                      gbt                  
rule                  1/1     3/5   10/20      1/1      3/5    10/20
attack                                                              
active_wiretap     1264.6  1236.4  1250.3  47588.2  49131.8  49824.1
arp_mitm            520.0   474.9   470.8  21854.4  21544.6  22064.6
fuzzing              90.3    41.0     6.7     89.2      0.0      0.0
mirai_botnet         76.7     0.0     0.0  11280.4   7147.6   1184.0
os_scan              86.7    19.3     0.0      1.0      0.0      0.0
ssdp_flood           87.8    27.8     0.0   1867.8   1743.5   1290.5
ssl_renegotiation   100.5    42.4    43.9     79.8      0.0      0.0
syn_dos             653.7   475.5    65.9     30.3      1.0      0.0
video_injection      11.3    10.3    10.6      0.0      0.0      0.0

## 4. Keep the output

Select **Save Version > Save & Run All (Commit)**. The folder `supervised` then has `results.csv` and the test scores for each model and capture.

In [9]:
print(results.drop(columns=["rule"]).to_csv(index=False))

attack,model,target_fpr,threshold,k,n,detected,delay_packets,benign_packets_before_attack,false_alarms_before_attack,false_alarms_per_100k,packet_recall,roc_auc,pr_auc
arp_mitm,logreg,0.01,0.6889684837733635,1,1,True,8495.0,195000,2128,1091.2820512820513,0.04563806676492571,0.6889965875846928,0.8899419064257468
arp_mitm,logreg,0.01,0.6889684837733635,3,5,True,8498.0,195000,1864,955.8974358974358,0.04256630739247969,0.6889965875846928,0.8899419064257468
arp_mitm,logreg,0.01,0.6889684837733635,10,20,True,8525.0,195000,1778,911.7948717948718,0.04304741581039264,0.6889965875846928,0.8899419064257468
arp_mitm,logreg,0.001,0.7039600476227658,1,1,True,8495.0,195000,1014,520.0,0.02150056929707528,0.6889965875846928,0.8899419064257468
arp_mitm,logreg,0.001,0.7039600476227658,3,5,True,8519.0,195000,926,474.87179487179486,0.020185597831781445,0.6889965875846928,0.8899419064257468
arp_mitm,logreg,0.001,0.7039600476227658,10,20,True,8555.0,195000,918,470.7692307692308,0.02011661858493004,0.68899658